# Improving Random Forests by Smoothing
**ArXivist-generated reproduction notebook**
Paper: 10.1007/s10994-026-07077-z
Generated: 2026-07-26

This notebook walks through the key components of the implementation, runs a
small-scale training loop, and verifies that the setup matches the paper's
reported behavior on a mini-dataset.

In [ ]:
# Check Python version, GPU availability, and key dependencies
import sys, torch
print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU — training will be slow")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Install the project in editable mode (run once)
import subprocess
result = subprocess.run(["pip", "install", "-e", ".."], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else result.stderr)

## Paper Overview
This paper introduces a probabilistic kernel smoothing approach to random forest regression. Random forests typically produce piecewise constant predictions, ignoring potential smoothness in the underlying function. By using probabilistic smoothing, the predictions of individual trees are integrated over a kernel, allowing information sharing across the input space.

The implementation includes:
- `random_forest.py`: A wrapper around scikit-learn's Random Forest to extract tree regions and predictions.
- `kernels.py`: Gaussian and Hyperbolic Secant kernels.
- `smoothed_rf.py`: The main `SmoothedRandomForest` module computing the smoothed predictions using out-of-bag optimization.

## Component 1: Base Random Forest
The base Random Forest wrapper extracts tree structures into PyTorch tensors for efficient parallel evaluation.

Equation:
$$ \hat{f}_{RF}(x_0 | \mathcal{F}, X, y) = \frac{1}{T} \sum_{t=1}^T \hat{f}(x_0 | T_t, m_t, X, y) $$

In [ ]:
import torch
import numpy as np
from src.improving_rf_smoothing.models.random_forest import RandomForestWrapper

# Instantiate base RF
model_config = {
    "n_estimators": 5,
    "max_depth": 3,
    "max_features": 1.0,
    "random_state": 42
}
base_rf = RandomForestWrapper(**model_config)

# Toy dataset
X_toy = np.random.randn(20, 4)
y_toy = np.random.randn(20)

# Fit the base RF to extract tree structures
base_rf.fit(X_toy, y_toy)
print(f"Base RF fitted with {base_rf.n_estimators} estimators.")

## Component 2: Smoothed Random Forest
The `SmoothedRandomForest` applies a kernel (e.g., Gaussian) to the extracted tree regions. The smoothing parameters $\Sigma, \beta_0, \beta_1$ are optimized.

Equation:
$$ \tilde{f}(x_0| X, y, \Sigma) = \mathbb{E}_{z \sim k(x_0, \Sigma)} [f(z | X, y)] $$
$$ \tilde{y}(x_0| X, y, \Sigma, \beta_0, \beta_1) = \beta_1 \tilde{f}(x_0| X, y, \Sigma) + \beta_0 $$

In [ ]:
from src.improving_rf_smoothing.models.smoothed_rf import SmoothedRandomForest
import torch

# Instantiate smoothed RF with the fitted base RF
smoothed_rf = SmoothedRandomForest(base_rf, variant="EST-PD", kernel="gaussian")

# Toy forward pass
try:
    x_input = torch.randn(2, 4).to(device)  # Batch of 2, 4 features
    smoothed_rf = smoothed_rf.to(device)
    output = smoothed_rf(x_input)
    print(f"Input shape:  {x_input.shape}")
    print(f"Output shape: {output.shape}")
    print(f"Expected:     [2, 1]")
except Exception as e:
    print(f"Error during forward pass: {e}")

## Mini-Training Demonstration
We will run a small-scale optimization of the smoothing parameters using the Out-Of-Bag (OOB) loss.

In [ ]:
# 1. Data cell: Generate synthetic data
import torch
from torch.utils.data import DataLoader, TensorDataset

X_train = torch.randn(100, 4)
y_train = X_train[:, 0] * 2 + torch.sin(X_train[:, 1]) + torch.randn(100) * 0.1

dataset = TensorDataset(X_train, y_train)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
print(f"Dataset created with {len(dataset)} samples.")

In [ ]:
# 2. Model init cell
from src.improving_rf_smoothing.models.random_forest import RandomForestWrapper
from src.improving_rf_smoothing.models.smoothed_rf import SmoothedRandomForest

base_rf = RandomForestWrapper(n_estimators=10, max_depth=5, random_state=42)
base_rf.fit(X_train.numpy(), y_train.numpy())

model = SmoothedRandomForest(base_rf, variant="EST", kernel="gaussian").to(device)
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model instantiated with {param_count} trainable parameters.")

In [ ]:
# 3. Training loop cell
import torch.optim as optim

optimizer = optim.AdamW(model.parameters(), lr=0.01)

print("Starting mini-training loop...")
model.train()
for step in range(10):
    total_loss = 0
    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        
        # Out-of-bag loss approximation for demo (in actual code, this requires OOB samples)
        # Using simple MSE for demonstration purposes
        preds = model(batch_X).squeeze()
        loss = torch.nn.functional.mse_loss(preds, batch_y)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    print(f"Step {step+1}/10, Loss: {total_loss / len(dataloader):.4f}")

In [ ]:
# 4. Results cell
model.eval()
with torch.no_grad():
    sample_x = X_train[:5].to(device)
    sample_y = y_train[:5].to(device)
    preds = model(sample_x).squeeze()
    
    print("Sample predictions vs targets:")
    for p, t in zip(preds, sample_y):
        print(f"Pred: {p.item():.4f} | Target: {t.item():.4f}")

## Paper Results Comparison

In [ ]:
# Results reported in the paper (from SIR evaluation_protocol.reported_results)
paper_results = {
    "dataset": "CCPP",
    "metric": "MSE",
    "reported_value": 30.88,
    "baseline": "RF(100)",
    "baseline_value": "Not explicitly listed"
}
print("Paper's claimed results:")
for k, v in paper_results.items():
    print(f"  {k}: {v}")

print("\nTo reproduce these results, run train.py with the full config.")
print("Then use the Results Comparator (Stage 6) to compare your outputs.")

## What to do next

1. **Full training**: `python train.py --config configs/config.yaml`
2. **Evaluation**: `python evaluate.py --checkpoint checkpoints/best.pt`
3. **Compare results**: Feed your results back to ArXivist's Results Comparator

**Implementation notes from the SIR:**
- Random forest hyperparameters not explicitly specified follow scikit-learn defaults. (Confidence: 0.95)
- Grid search used for max_depth is a typical range (e.g., 5-30). (Confidence: 0.7)